<a href="https://colab.research.google.com/github/sowrin-paul/flyrank-intern/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

Lane 2 - Refresh / Content Opportunity Scoring.
I'm picking this over pure signal analysis (Lane 1) because FlyRank's reviewers don't need "what correlates with decline" - they need an ordered list of which pages to open first. A ranking output maps directly onto a real, limited-capacity decision, which is what this internship is graded on. The starter data also has real teeth for it: 43.8% of qualifying pages already show real search demand paired with a falling trend, and another third show a completely different fixable problem (weak CTR, not decline) - so a single flag isn't enough, which is exactly the kind of "too messy for one if-statement" problem a ranked, reason-coded model is for.

In [5]:
# lane note only, no code needed here

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

Unit of analysis: one content item (content_id) per client, described by its trailing-90-day metrics — not a day, not a client. (A future week may narrow this to content-item forward-30-day-window once I define a real future-outcome label.)

Decision it improves: how a reviewer with limited hours allocates their attention across a client's page inventory — refresh, expand, fix CTR/metadata, protect, or leave alone.

Who acts, and how: a content/SEO reviewer on the client-facing team. They work down my ranked queue instead of guessing, and the reason code (e.g. declining_with_demand, low_ctr_visible_page) tells them what to actually do to the page, not just that it's "bad."

Output: a ranked review queue - page id, score, and reason code.

Cost of a wrong call:

False positive (I rank a page high but it isn't really a problem): a reviewer burns limited hours on a page that didn't need it, while a real problem page waits.
False negative (a genuinely declining, high-demand page isn't surfaced): the client keeps losing visibility/clicks silently until someone notices by accident.
Because review hours are the scarce resource, precision in the top of the queue (precision@K) matters more than overall accuracy — a wrong page in slot #3 is more expensive than a wrong page in slot #300.

Why data/ML, not a hand-written rule: no single column separates "worth reviewing" from "fine." A page can look fine on trend but be leaking clicks on CTR, or look thin but have no real demand behind it. 32.5% of pages qualify as low-CTR opportunities and a mostly-different 43.8% qualify as declining-with-demand — two real, overlapping, weakly-correlated problem types across 30,000 pages and 32 clients is exactly the "many signals, tangled" case a plain rule can't hand-tune, but a scored/ranked model can weigh and a human can still audit through reason codes.

In [6]:
# no code needed here

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [7]:
import pandas as pd

df = pd.read_csv("/content/content_refresh_anonymized.csv")

# starter-pipeline filter: only pages with real demand and enough age to trust the label
f = df[(df.impressions_90d > 0) & (df.content_age_days >= 90)].drop_duplicates("content_id")

print(f"Qualifying pages: {len(f):,} across {f.client_id.nunique()} clients")

declining_with_demand = (f.trend_direction == "down") & (f.impressions_90d >= 100)
low_ctr_visible = (f.impressions_90d >= 500) & (f.avg_position.between(0.01, 20)) & (f.ctr < 0.5)

print(f"declining_with_demand: {declining_with_demand.sum():,} pages "
      f"({declining_with_demand.mean()*100:.1f}% of qualifying pages)")
print(f"low_ctr_visible_page: {low_ctr_visible.sum():,} pages "
      f"({low_ctr_visible.mean()*100:.1f}% of qualifying pages)")
print(f"Overlap between the two reason codes: "
      f"{(declining_with_demand & low_ctr_visible).sum():,} pages")

Qualifying pages: 30,000 across 32 clients
declining_with_demand: 13,152 pages (43.8% of qualifying pages)
low_ctr_visible_page: 9,759 pages (32.5% of qualifying pages)
Overlap between the two reason codes: 6,120 pages


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

What this work can say: observed associations between page signals (trend, CTR, position, freshness) and a defined proxy or future outcome; a decision-support ranking that helps a reviewer prioritize limited hours; directional statements like "pages with X pattern are more often flagged for review" backed by precision@K on held-out clients.

What this work cannot say: that refreshing a page caused a recovery; that I've decoded any part of Google's ranking algorithm; that the starter label *(trend_direction == "down", a current-window bucket)* is a proven future outcome — it's a beginner proxy I plan to replace with a real forward-looking window once I move past the starter dataset. I also won't publish or reference any client names, URLs, domains, or raw queries — only pseudonymized ids and aggregated numbers.

In [8]:
# no code needed here

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.